# Pré-processamento de Dados

Este notebook realiza a limpeza e transformação dos dados e constrói
um pipeline de pré-processamento para o conjunto de dados de diabetes.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


## Preparação do Ambiente

Aqui importamos as bibliotecas necessárias para:
- carregar e manipular o dataset (`pandas`, `numpy`)
- dividir os dados em treino/validação/teste (`train_test_split`)
- criar um pipeline de transformação numérica com padronização (`StandardScaler`, `Pipeline`, `ColumnTransformer`)

A ideia é organizar o pré-processamento de forma reprodutível e sem vazamento de dados.


In [ ]:
df = pd.read_csv("../data/raw/diabetes.csv")

## Carregamento do Dataset

Nesta etapa, carregamos o CSV bruto. A partir daqui, todas as transformações (tratamento de valores inválidos, imputação, split e scaling) serão aplicadas sobre esse `DataFrame`.


In [5]:
cols_with_zero = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[cols_with_zero] = df[cols_with_zero].replace(0, np.nan)

df.isna().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

## Tratamento de valores inválidos (zeros como ausentes)

Algumas colunas biométricas não deveriam ter valor **0** (ex.: `Glucose`, `BMI`).
Neste dataset, esses zeros geralmente significam *ausência de medição*.

Estratégia:
1. substituir 0 por `NaN` apenas nas colunas onde isso faz sentido
2. inspecionar a quantidade de ausentes por coluna (`df.isna().sum()`)


In [6]:
df[cols_with_zero] = df[cols_with_zero].fillna(df[cols_with_zero].median())
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

## Imputação (preenchimento de ausentes)

Para não perder linhas do dataset, aplicamos uma imputação simples:
- preencher `NaN` pela **mediana** de cada coluna

A mediana é uma escolha robusta a outliers e costuma funcionar bem como baseline.


In [7]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

## Separação entre features (X) e alvo (y)

Aqui separamos:
- `X`: variáveis explicativas (todas as colunas exceto `Outcome`)
- `y`: variável alvo (`Outcome`)

Essa separação facilita o split e evita misturar o alvo em transformações de features.


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

## Divisão em treino / validação / teste

Fazemos um split em duas etapas para obter três subconjuntos:
- **treino** (fit do modelo)
- **validação** (tuning/seleção)
- **teste** (avaliação final)

Usamos `stratify` para manter a proporção de classes semelhante em cada conjunto.


In [ ]:
numeric_features = X.columns

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ]
)

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)


## Pipeline de pré-processamento (padronização)

Como todas as features aqui são numéricas, aplicamos `StandardScaler` para colocar as variáveis na mesma escala (média 0, desvio 1).

Ponto importante: o `fit` do scaler é feito **somente no treino** (`fit_transform(X_train)`), e depois aplicamos apenas `transform` em validação e teste. Isso evita vazamento de informação.


### Saídas do pré-processamento

Ao final, `X_train_scaled`, `X_val_scaled` e `X_test_scaled` são matrizes numéricas prontas para alimentar modelos do scikit-learn.

Próximo passo típico:
- treinar modelos em `X_train_scaled`/`y_train`
- validar em `X_val_scaled`/`y_val`
- reportar resultado final em `X_test_scaled`/`y_test`


## Conclusão

Os dados foram devidamente limpos e preparados para a etapa de modelagem.
Foi construído um pipeline de pré-processamento que garante consistência,
reprodutibilidade e evita vazamento de dados.
